# Fase 3 – Procesamiento, modelado y experimentación

## Integración de datasets ambientales y meteorológicos

**Proyecto:** Identificación de zonas críticas de contaminación atmosférica mediante técnicas GIS y aprendizaje automático utilizando concentraciones de PM2.5 registradas en la Ciudad de México durante el periodo 2024–2025.

**Responsable técnico:** Miranda Patricia Pérez Camelo

## Objetivo

Integrar los datasets limpios de PM2.5, temperatura, humedad relativa, velocidad del viento, dirección del viento y catálogo de estaciones, generando una base consolidada que pueda utilizarse posteriormente para el análisis estadístico, temporal, espacial y el modelado mediante aprendizaje automático.

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
CARPETA_PROYECTO = Path("..")

CARPETA_DATOS = (
    CARPETA_PROYECTO
    / "datos"
    / "datos_limpios"
    / "dataset_limpios"
    / "dataset_limpios"
)

CARPETA_RESULTADOS = CARPETA_PROYECTO / "resultados"

CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos:", CARPETA_DATOS.resolve())
print("Carpeta de resultados:", CARPETA_RESULTADOS.resolve())

Carpeta de datos: C:\Users\pmiri\OneDrive\Desktop\ProyAmbiental\datos\datos_limpios\dataset_limpios\dataset_limpios
Carpeta de resultados: C:\Users\pmiri\OneDrive\Desktop\ProyAmbiental\resultados


In [3]:
archivos_requeridos = [
    CARPETA_DATOS / "PM25_24_25" / "24PM25_limpio.csv",
    CARPETA_DATOS / "PM25_24_25" / "25PM25_limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "TMP_2024_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "TMP_2025_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "RH_2024_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "RH_2025_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "WSP_2024_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "WSP_2025_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "WDR_2024_Limpio.csv",
    CARPETA_DATOS / "Meteorologia_24_25" / "WDR_2025_Limpio.csv",
    CARPETA_DATOS / "Catalogo_Estaciones.csv",
]

for archivo in archivos_requeridos:
    estado = "ENCONTRADO" if archivo.exists() else "NO ENCONTRADO"
    print(f"{estado}: {archivo.name}")

ENCONTRADO: 24PM25_limpio.csv
ENCONTRADO: 25PM25_limpio.csv
ENCONTRADO: TMP_2024_Limpio.csv
ENCONTRADO: TMP_2025_Limpio.csv
ENCONTRADO: RH_2024_Limpio.csv
ENCONTRADO: RH_2025_Limpio.csv
ENCONTRADO: WSP_2024_Limpio.csv
ENCONTRADO: WSP_2025_Limpio.csv
ENCONTRADO: WDR_2024_Limpio.csv
ENCONTRADO: WDR_2025_Limpio.csv
ENCONTRADO: Catalogo_Estaciones.csv


## 1. Transformación de formato ancho a formato largo

Los datasets originales utilizan una columna para cada estación de monitoreo. Para facilitar su integración, se transformarán a formato largo, de modo que cada fila represente una medición realizada en una estación, fecha y hora determinadas.

In [4]:
def cargar_y_transformar(
    ruta_archivo: Path,
    nombre_variable: str,
    anio: int
) -> pd.DataFrame:
    """
    Lee un CSV en formato ancho y lo transforma a formato largo.

    Parámetros
    ----------
    ruta_archivo:
        Ruta del archivo CSV.
    nombre_variable:
        Nombre que tendrá la columna de valores, por ejemplo PM25 o TMP.
    anio:
        Año correspondiente al archivo.

    Retorna
    -------
    DataFrame con columnas:
    FECHA, HORA, ESTACION y la variable indicada.
    """

    if not ruta_archivo.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta_archivo}")

    df = pd.read_csv(ruta_archivo)

    columnas_obligatorias = {"FECHA", "HORA"}

    if not columnas_obligatorias.issubset(df.columns):
        raise ValueError(
            f"{ruta_archivo.name} no contiene FECHA y HORA."
        )

    df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")
    df["HORA"] = pd.to_numeric(df["HORA"], errors="coerce")

    df_largo = df.melt(
        id_vars=["FECHA", "HORA"],
        var_name="ESTACION",
        value_name=nombre_variable
    )

    df_largo["ANIO"] = anio
    df_largo["ESTACION"] = (
        df_largo["ESTACION"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    return df_largo

## 2. Integración temporal de PM2.5

Se transformarán los archivos correspondientes a 2024 y 2025 y posteriormente se unirán verticalmente para obtener una sola base de concentraciones de PM2.5.

In [5]:
pm25_2024 = cargar_y_transformar(
    CARPETA_DATOS / "PM25_24_25" / "24PM25_limpio.csv",
    "PM25",
    2024
)

pm25_2024.head()

,FECHA,HORA,ESTACION,PM25,ANIO
0,2024-01-01,1,AJM,33.0,2024
1,2024-01-01,2,AJM,21.0,2024
2,2024-01-01,3,AJM,18.0,2024
3,2024-01-01,4,AJM,17.0,2024
4,2024-01-01,5,AJM,18.0,2024


In [6]:
print("Dimensiones PM2.5 2024:", pm25_2024.shape)
print(pm25_2024.head())

Dimensiones PM2.5 2024: (210768, 5)
       FECHA  HORA ESTACION  PM25  ANIO
0 2024-01-01     1      AJM  33.0  2024
1 2024-01-01     2      AJM  21.0  2024
2 2024-01-01     3      AJM  18.0  2024
3 2024-01-01     4      AJM  17.0  2024
4 2024-01-01     5      AJM  18.0  2024


In [7]:
pm25_2025 = cargar_y_transformar(
    CARPETA_DATOS / "PM25_24_25" / "25PM25_limpio.csv",
    "PM25",
    2025
)

print("Dimensiones PM2.5 2025:", pm25_2025.shape)

pm25_2025.head()

Dimensiones PM2.5 2025: (192720, 5)


,FECHA,HORA,ESTACION,PM25,ANIO
0,2025-01-01,1,AJM,13.0,2025
1,2025-01-01,2,AJM,12.0,2025
2,2025-01-01,3,AJM,11.0,2025
3,2025-01-01,4,AJM,14.0,2025
4,2025-01-01,5,AJM,15.0,2025


In [8]:
pm25 = pd.concat(
    [pm25_2024, pm25_2025],
    ignore_index=True
)

print("Dimensiones de PM2.5 combinado:", pm25.shape)
print("Años disponibles:", sorted(pm25["ANIO"].unique()))

pm25.head()

Dimensiones de PM2.5 combinado: (403488, 5)
Años disponibles: [np.int64(2024), np.int64(2025)]


,FECHA,HORA,ESTACION,PM25,ANIO
0,2024-01-01,1,AJM,33.0,2024
1,2024-01-01,2,AJM,21.0,2024
2,2024-01-01,3,AJM,18.0,2024
3,2024-01-01,4,AJM,17.0,2024
4,2024-01-01,5,AJM,18.0,2024


In [9]:
llave = ["FECHA", "HORA", "ESTACION"]

duplicados_pm25 = pm25.duplicated(subset=llave).sum()

print("Registros duplicados en PM2.5:", duplicados_pm25)

Registros duplicados en PM2.5: 0


In [11]:
print("Total de filas:", len(pm25))
print("PM2.5 con valor:", pm25["PM25"].notna().sum())
print("PM2.5 faltante:", pm25["PM25"].isna().sum())
print("Estaciones:", pm25["ESTACION"].nunique())

Total de filas: 403488
PM2.5 con valor: 206858
PM2.5 faltante: 196630
Estaciones: 24


## 3. Transformación e integración de variables meteorológicas

Se procesarán los archivos de temperatura, humedad relativa, velocidad del viento y dirección del viento correspondientes a 2024 y 2025. Cada variable será transformada a formato largo y consolidada temporalmente.

In [12]:
def cargar_variable_dos_anios(
    nombre_variable: str,
    archivo_2024: str,
    archivo_2025: str
) -> pd.DataFrame:
    """
    Carga, transforma y une una variable meteorológica
    correspondiente a 2024 y 2025.
    """

    carpeta_meteorologia = (
        CARPETA_DATOS / "Meteorologia_24_25"
    )

    df_2024 = cargar_y_transformar(
        carpeta_meteorologia / archivo_2024,
        nombre_variable,
        2024
    )

    df_2025 = cargar_y_transformar(
        carpeta_meteorologia / archivo_2025,
        nombre_variable,
        2025
    )

    df_completo = pd.concat(
        [df_2024, df_2025],
        ignore_index=True
    )

    return df_completo

In [14]:
tmp = cargar_variable_dos_anios(
    "TMP",
    "TMP_2024_Limpio.csv",
    "TMP_2025_Limpio.csv"
)

print("Temperatura:", tmp.shape)
tmp.head()

Temperatura: (491008, 5)


,FECHA,HORA,ESTACION,TMP,ANIO
0,2024-01-01,1,ACO,NaN,2024
1,2024-01-01,2,ACO,NaN,2024
2,2024-01-01,3,ACO,NaN,2024
3,2024-01-01,4,ACO,NaN,2024
4,2024-01-01,5,ACO,NaN,2024


In [15]:
rh = cargar_variable_dos_anios(
    "RH",
    "RH_2024_Limpio.csv",
    "RH_2025_Limpio.csv"
)

print("Humedad relativa:", rh.shape)
rh.head()

Humedad relativa: (491008, 5)


,FECHA,HORA,ESTACION,RH,ANIO
0,2024-01-01,1,ACO,NaN,2024
1,2024-01-01,2,ACO,NaN,2024
2,2024-01-01,3,ACO,NaN,2024
3,2024-01-01,4,ACO,NaN,2024
4,2024-01-01,5,ACO,NaN,2024


In [16]:
wsp = cargar_variable_dos_anios(
    "WSP",
    "WSP_2024_Limpio.csv",
    "WSP_2025_Limpio.csv"
)

print("Velocidad del viento:", wsp.shape)
wsp.head()

Velocidad del viento: (491008, 5)


,FECHA,HORA,ESTACION,WSP,ANIO
0,2024-01-01,1,ACO,NaN,2024
1,2024-01-01,2,ACO,NaN,2024
2,2024-01-01,3,ACO,NaN,2024
3,2024-01-01,4,ACO,NaN,2024
4,2024-01-01,5,ACO,NaN,2024


In [17]:
wdr = cargar_variable_dos_anios(
    "WDR",
    "WDR_2024_Limpio.csv",
    "WDR_2025_Limpio.csv"
)

print("Dirección del viento:", wdr.shape)
wdr.head()

Dirección del viento: (491008, 5)


,FECHA,HORA,ESTACION,WDR,ANIO
0,2024-01-01,1,ACO,NaN,2024
1,2024-01-01,2,ACO,NaN,2024
2,2024-01-01,3,ACO,NaN,2024
3,2024-01-01,4,ACO,NaN,2024
4,2024-01-01,5,ACO,NaN,2024


In [18]:
datasets_revision = {
    "PM25": pm25,
    "TMP": tmp,
    "RH": rh,
    "WSP": wsp,
    "WDR": wdr
}

for nombre, df in datasets_revision.items():
    duplicados = df.duplicated(subset=llave).sum()

    print(
        f"{nombre}: {duplicados} registros duplicados "
        f"en FECHA-HORA-ESTACION"
    )

PM25: 0 registros duplicados en FECHA-HORA-ESTACION
TMP: 0 registros duplicados en FECHA-HORA-ESTACION
RH: 0 registros duplicados en FECHA-HORA-ESTACION
WSP: 0 registros duplicados en FECHA-HORA-ESTACION
WDR: 0 registros duplicados en FECHA-HORA-ESTACION


In [19]:
estaciones_por_dataset = {
    nombre: set(df["ESTACION"].dropna().unique())
    for nombre, df in datasets_revision.items()
}

estaciones_comunes = set.intersection(
    *estaciones_por_dataset.values()
)

print("Cantidad de estaciones comunes:", len(estaciones_comunes))
print("Estaciones comunes:")
print(sorted(estaciones_comunes))

Cantidad de estaciones comunes: 20
Estaciones comunes:
['AJM', 'AJU', 'BJU', 'FAR', 'GAM', 'HGM', 'INN', 'MER', 'MGH', 'MON', 'MPA', 'NEZ', 'PED', 'SAC', 'SAG', 'SFE', 'TLA', 'UAX', 'UIZ', 'XAL']


## 4. Integración ambiental y meteorológica

La integración se realizará utilizando una llave compuesta por FECHA, HORA y ESTACION. PM2.5 se utilizará como base principal y las variables meteorológicas se añadirán cuando exista una coincidencia temporal y espacial.

In [21]:
tmp_reducido = tmp[llave + ["TMP"]]
rh_reducido = rh[llave + ["RH"]]
wsp_reducido = wsp[llave + ["WSP"]]
wdr_reducido = wdr[llave + ["WDR"]]

In [22]:
datos_integrados = (
    pm25
    .merge(
        tmp_reducido,
        on=llave,
        how="left",
        validate="one_to_one"
    )
    .merge(
        rh_reducido,
        on=llave,
        how="left",
        validate="one_to_one"
    )
    .merge(
        wsp_reducido,
        on=llave,
        how="left",
        validate="one_to_one"
    )
    .merge(
        wdr_reducido,
        on=llave,
        how="left",
        validate="one_to_one"
    )
)

print("Integración ambiental y meteorológica completada")
print("Dimensiones:", datos_integrados.shape)

datos_integrados.head()

Integración ambiental y meteorológica completada
Dimensiones: (403488, 9)


,FECHA,HORA,ESTACION,PM25,ANIO,TMP,RH,WSP,WDR
0,2024-01-01,1,AJM,33.0,2024,6.0,57.0,3.5,320.0
1,2024-01-01,2,AJM,21.0,2024,5.0,60.0,NaN,NaN
2,2024-01-01,3,AJM,18.0,2024,4.0,63.0,3.9,314.0
3,2024-01-01,4,AJM,17.0,2024,4.0,64.0,3.7,320.0
4,2024-01-01,5,AJM,18.0,2024,3.0,65.0,3.2,324.0


In [23]:
print("Filas antes de integrar:", len(pm25))
print("Filas después de integrar:", len(datos_integrados))

if len(pm25) == len(datos_integrados):
    print("La integración no duplicó registros.")
else:
    print("Advertencia: cambió el número de filas.")

Filas antes de integrar: 403488
Filas después de integrar: 403488
La integración no duplicó registros.


In [24]:
columnas_analisis = ["PM25", "TMP", "RH", "WSP", "WDR"]

resumen_disponibilidad = pd.DataFrame({
    "registros_con_dato": datos_integrados[columnas_analisis].notna().sum(),
    "registros_faltantes": datos_integrados[columnas_analisis].isna().sum(),
    "porcentaje_disponible": (
        datos_integrados[columnas_analisis].notna().mean() * 100
    ).round(2)
})

resumen_disponibilidad

,registros_con_dato,registros_faltantes,porcentaje_disponible
PM25,206858,196630,51.27
TMP,191304,212184,47.41
RH,228288,175200,56.58
WSP,212191,191297,52.59
WDR,212196,191292,52.59


## 5. Incorporación de información geográfica

Se añadirá la información del catálogo de estaciones para asociar cada registro ambiental con el nombre, longitud, latitud y altitud de su estación de monitoreo.

In [25]:
catalogo = pd.read_csv(
    CARPETA_DATOS / "Catalogo_Estaciones.csv"
)

catalogo["cve_estac"] = (
    catalogo["cve_estac"]
    .astype(str)
    .str.strip()
    .str.upper()
)

catalogo.head()

,cve_estac,nom_estac,longitud,latitud,alt,obs_estac,id_station
0,ACO,Acolman,-98.912003,19.635501,2198.0,NaN,484150020109
1,AJU,Ajusco,-99.162611,19.154286,2942.0,NaN,484090120400
2,AJM,Ajusco Medio,-99.207744,19.272161,2548.0,NaN,484090120609
3,ARA,Aragón,-99.074549,19.470218,2200.0,Finalizó operación en 2010,484090050301
4,ATI,Atizapan,-99.254133,19.576963,2341.0,NaN,484150130101


In [26]:
duplicados_catalogo = catalogo["cve_estac"].duplicated().sum()

print(
    "Claves de estación duplicadas en el catálogo:",
    duplicados_catalogo
)

Claves de estación duplicadas en el catálogo: 0


In [27]:
catalogo_reducido = catalogo[
    [
        "cve_estac",
        "nom_estac",
        "longitud",
        "latitud",
        "alt",
        "id_station"
    ]
].copy()

In [28]:
datos_integrados = datos_integrados.merge(
    catalogo_reducido,
    left_on="ESTACION",
    right_on="cve_estac",
    how="left",
    validate="many_to_one"
)

print("Catálogo integrado correctamente")
print("Dimensiones:", datos_integrados.shape)

datos_integrados.head()

Catálogo integrado correctamente
Dimensiones: (403488, 15)


,FECHA,HORA,ESTACION,PM25,ANIO,TMP,RH,WSP,WDR,cve_estac,nom_estac,longitud,latitud,alt,id_station
0,2024-01-01,1,AJM,33.0,2024,6.0,57.0,3.5,320.0,AJM,Ajusco Medio,-99.207744,19.272161,2548.0,484090120609
1,2024-01-01,2,AJM,21.0,2024,5.0,60.0,NaN,NaN,AJM,Ajusco Medio,-99.207744,19.272161,2548.0,484090120609
2,2024-01-01,3,AJM,18.0,2024,4.0,63.0,3.9,314.0,AJM,Ajusco Medio,-99.207744,19.272161,2548.0,484090120609
3,2024-01-01,4,AJM,17.0,2024,4.0,64.0,3.7,320.0,AJM,Ajusco Medio,-99.207744,19.272161,2548.0,484090120609
4,2024-01-01,5,AJM,18.0,2024,3.0,65.0,3.2,324.0,AJM,Ajusco Medio,-99.207744,19.272161,2548.0,484090120609


In [29]:
sin_coordenadas = (
    datos_integrados.loc[
        datos_integrados["latitud"].isna()
        | datos_integrados["longitud"].isna(),
        "ESTACION"
    ]
    .drop_duplicates()
    .sort_values()
)

print("Estaciones sin coordenadas:")
print(sin_coordenadas.tolist())

Estaciones sin coordenadas:
[]


## 6. Construcción de la variable temporal

Se construirá una variable DATETIME combinando FECHA y HORA. Esta columna permitirá ordenar los registros, analizar series temporales y generar variables como mes, día de la semana y hora.

In [30]:
datos_integrados["DATETIME"] = (
    pd.to_datetime(
        datos_integrados["FECHA"],
        errors="coerce"
    )
    + pd.to_timedelta(
        datos_integrados["HORA"] - 1,
        unit="h"
    )
)

In [31]:
datos_integrados["ANIO"] = datos_integrados["DATETIME"].dt.year
datos_integrados["MES"] = datos_integrados["DATETIME"].dt.month
datos_integrados["DIA"] = datos_integrados["DATETIME"].dt.day
datos_integrados["DIA_SEMANA"] = datos_integrados["DATETIME"].dt.dayofweek
datos_integrados["HORA_DIA"] = datos_integrados["DATETIME"].dt.hour

datos_integrados[
    [
        "FECHA",
        "HORA",
        "DATETIME",
        "ANIO",
        "MES",
        "DIA_SEMANA",
        "HORA_DIA"
    ]
].head()

,FECHA,HORA,DATETIME,ANIO,MES,DIA_SEMANA,HORA_DIA
0,2024-01-01,1,2024-01-01 00:00:00,2024,1,0,0
1,2024-01-01,2,2024-01-01 01:00:00,2024,1,0,1
2,2024-01-01,3,2024-01-01 02:00:00,2024,1,0,2
3,2024-01-01,4,2024-01-01 03:00:00,2024,1,0,3
4,2024-01-01,5,2024-01-01 04:00:00,2024,1,0,4


In [32]:
print(
    "Registros sin DATETIME válido:",
    datos_integrados["DATETIME"].isna().sum()
)

print(
    "Fecha mínima:",
    datos_integrados["DATETIME"].min()
)

print(
    "Fecha máxima:",
    datos_integrados["DATETIME"].max()
)

Registros sin DATETIME válido: 0
Fecha mínima: 2024-01-01 00:00:00
Fecha máxima: 2025-12-31 23:00:00


In [33]:
columnas_finales = [
    "DATETIME",
    "FECHA",
    "HORA",
    "ANIO",
    "MES",
    "DIA",
    "DIA_SEMANA",
    "HORA_DIA",
    "ESTACION",
    "nom_estac",
    "PM25",
    "TMP",
    "RH",
    "WSP",
    "WDR",
    "longitud",
    "latitud",
    "alt",
    "id_station"
]

datos_integrados = datos_integrados[columnas_finales]

In [34]:
datos_integrados = (
    datos_integrados
    .sort_values(
        by=["DATETIME", "ESTACION"]
    )
    .reset_index(drop=True)
)

datos_integrados.head()

,DATETIME,FECHA,HORA,ANIO,MES,DIA,DIA_SEMANA,HORA_DIA,ESTACION,nom_estac,PM25,TMP,RH,WSP,WDR,longitud,latitud,alt,id_station
0,2024-01-01,2024-01-01,1,2024,1,1,0,0,AJM,Ajusco Medio,33.0,6.0,57.0,3.5,320.0,-99.207744,19.272161,2548.0,484090120609
1,2024-01-01,2024-01-01,1,2024,1,1,0,0,AJU,Ajusco,NaN,1.0,99.0,0.9,54.0,-99.162611,19.154286,2942.0,484090120400
2,2024-01-01,2024-01-01,1,2024,1,1,0,0,BJU,Benito Juárez,42.0,12.0,56.0,0.8,173.0,-99.159596,19.370464,2249.0,484090140201
3,2024-01-01,2024-01-01,1,2024,1,1,0,0,CAM,Camarones,46.0,NaN,NaN,NaN,NaN,-99.169794,19.468404,2233.0,484090020301
4,2024-01-01,2024-01-01,1,2024,1,1,0,0,CCA,Centro de Ciencias de la Atmósfera,NaN,NaN,NaN,NaN,NaN,-99.176111,19.326111,2294.0,484090030501


In [35]:
print("Dimensiones finales:", datos_integrados.shape)
print("Columnas finales:")
print(datos_integrados.columns.tolist())

datos_integrados.info()

Dimensiones finales: (403488, 19)
Columnas finales:
['DATETIME', 'FECHA', 'HORA', 'ANIO', 'MES', 'DIA', 'DIA_SEMANA', 'HORA_DIA', 'ESTACION', 'nom_estac', 'PM25', 'TMP', 'RH', 'WSP', 'WDR', 'longitud', 'latitud', 'alt', 'id_station']
<class 'pandas.DataFrame'>
RangeIndex: 403488 entries, 0 to 403487
Data columns (total 19 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   DATETIME    403488 non-null  datetime64[us]
 1   FECHA       403488 non-null  datetime64[us]
 2   HORA        403488 non-null  int64         
 3   ANIO        403488 non-null  int32         
 4   MES         403488 non-null  int32         
 5   DIA         403488 non-null  int32         
 6   DIA_SEMANA  403488 non-null  int32         
 7   HORA_DIA    403488 non-null  int32         
 8   ESTACION    403488 non-null  str           
 9   nom_estac   403488 non-null  str           
 10  PM25        206858 non-null  float64       
 11  TMP         191304 non-n

In [36]:
datos_analisis = datos_integrados.dropna(
    subset=["PM25"]
).copy()

print("Filas del dataset completo:", len(datos_integrados))
print("Filas con PM2.5 disponible:", len(datos_analisis))

Filas del dataset completo: 403488
Filas con PM2.5 disponible: 206858


In [37]:
variables_modelo = [
    "PM25",
    "TMP",
    "RH",
    "WSP",
    "WDR",
    "latitud",
    "longitud"
]

datos_modelo_completos = datos_integrados.dropna(
    subset=variables_modelo
).copy()

print(
    "Filas con todas las variables disponibles:",
    len(datos_modelo_completos)
)

Filas con todas las variables disponibles: 108060


In [38]:
duplicados_finales = datos_integrados.duplicated(
    subset=["DATETIME", "ESTACION"]
).sum()

print("Duplicados finales:", duplicados_finales)

Duplicados finales: 0


In [39]:
resumen_nulos_final = pd.DataFrame({
    "faltantes": datos_integrados.isna().sum(),
    "porcentaje": (
        datos_integrados.isna().mean() * 100
    ).round(2)
}).sort_values(
    by="porcentaje",
    ascending=False
)

resumen_nulos_final

,faltantes,porcentaje
TMP,212184,52.59
PM25,196630,48.73
WSP,191297,47.41
WDR,191292,47.41
RH,175200,43.42
DATETIME,0,0.00
FECHA,0,0.00
HORA,0,0.00
ANIO,0,0.00
DIA,0,0.00


In [40]:
validaciones = {
    "PM25 negativos": (datos_integrados["PM25"] < 0).sum(),
    "RH menor a 0": (datos_integrados["RH"] < 0).sum(),
    "RH mayor a 100": (datos_integrados["RH"] > 100).sum(),
    "WSP negativa": (datos_integrados["WSP"] < 0).sum(),
    "WDR menor a 0": (datos_integrados["WDR"] < 0).sum(),
    "WDR mayor a 360": (datos_integrados["WDR"] > 360).sum()
}

pd.Series(validaciones, name="cantidad")

PM25 negativos     0
RH menor a 0       3
RH mayor a 100     0
WSP negativa       0
WDR menor a 0      0
WDR mayor a 360    0
Name: cantidad, dtype: int64

In [41]:
ruta_salida_completa = (
    CARPETA_RESULTADOS
    / "dataset_integrado_2024_2025.csv"
)

datos_integrados.to_csv(
    ruta_salida_completa,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset completo guardado en:")
print(ruta_salida_completa.resolve())

Dataset completo guardado en:
C:\Users\pmiri\OneDrive\Desktop\ProyAmbiental\resultados\dataset_integrado_2024_2025.csv


In [42]:
ruta_salida_analisis = (
    CARPETA_RESULTADOS
    / "dataset_analisis_pm25_2024_2025.csv"
)

datos_analisis.to_csv(
    ruta_salida_analisis,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset de análisis guardado en:")
print(ruta_salida_analisis.resolve())

Dataset de análisis guardado en:
C:\Users\pmiri\OneDrive\Desktop\ProyAmbiental\resultados\dataset_analisis_pm25_2024_2025.csv


In [43]:
ruta_salida_modelo = (
    CARPETA_RESULTADOS
    / "dataset_modelo_completo_2024_2025.csv"
)

datos_modelo_completos.to_csv(
    ruta_salida_modelo,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset para modelado guardado en:")
print(ruta_salida_modelo.resolve())

Dataset para modelado guardado en:
C:\Users\pmiri\OneDrive\Desktop\ProyAmbiental\resultados\dataset_modelo_completo_2024_2025.csv


In [44]:
for ruta in [
    ruta_salida_completa,
    ruta_salida_analisis,
    ruta_salida_modelo
]:
    print(ruta.name, "→", ruta.exists())

dataset_integrado_2024_2025.csv → True
dataset_analisis_pm25_2024_2025.csv → True
dataset_modelo_completo_2024_2025.csv → True


## Conclusión de la integración

Se transformaron los datasets ambientales y meteorológicos de formato ancho a formato largo, permitiendo representar cada medición mediante una combinación de fecha, hora y estación.

Posteriormente se consolidaron los registros correspondientes a 2024 y 2025 y se integraron las concentraciones de PM2.5 con las variables de temperatura, humedad relativa, velocidad del viento y dirección del viento.

También se incorporó el catálogo de estaciones, añadiendo nombre, coordenadas geográficas, altitud e identificadores espaciales. Como resultado se generó un dataset integrado que servirá como base para los análisis temporales, estadísticos, espaciales y los experimentos de aprendizaje automático.

La integración conserva los registros de PM2.5 como fuente principal y permite identificar la disponibilidad de las variables meteorológicas para cada estación y momento de medición.